In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

In [2]:
df = pd.read_csv('../Datasets/preprocessed_liver_data.csv')

X = df.drop('Result', axis=1)
y = df['Result']

In [3]:
skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

xgb_acc = []
xgb_auc = []

In [4]:
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n--- XGBoost Fold {fold} ---")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # SMOTE only on training fold
    smote = SMOTE(random_state=42)
    X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

    # XGBoost model
    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    )

    # Train
    xgb.fit(X_train_bal, y_train_bal)

    # Predict
    y_pred = xgb.predict(X_test)
    y_prob = xgb.predict_proba(X_test)[:, 1]

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    xgb_acc.append(acc)
    xgb_auc.append(auc)

    print(f"Accuracy : {acc:.4f}")
    print(f"ROC-AUC  : {auc:.4f}")


--- XGBoost Fold 1 ---
Accuracy : 0.9954
ROC-AUC  : 0.9999

--- XGBoost Fold 2 ---
Accuracy : 0.9933
ROC-AUC  : 0.9997

--- XGBoost Fold 3 ---
Accuracy : 0.9959
ROC-AUC  : 0.9997

--- XGBoost Fold 4 ---
Accuracy : 0.9933
ROC-AUC  : 0.9998

--- XGBoost Fold 5 ---
Accuracy : 0.9917
ROC-AUC  : 0.9983

--- XGBoost Fold 6 ---
Accuracy : 0.9979
ROC-AUC  : 1.0000

--- XGBoost Fold 7 ---
Accuracy : 0.9933
ROC-AUC  : 0.9984

--- XGBoost Fold 8 ---
Accuracy : 0.9912
ROC-AUC  : 0.9995

--- XGBoost Fold 9 ---
Accuracy : 0.9943
ROC-AUC  : 0.9999

--- XGBoost Fold 10 ---
Accuracy : 0.9933
ROC-AUC  : 0.9986


In [5]:
print("\n====== XGBOOST (CV RESULTS) ======")
print(f"Accuracy : {np.mean(xgb_acc):.4f} ± {np.std(xgb_acc):.4f}")
print(f"ROC-AUC  : {np.mean(xgb_auc):.4f} ± {np.std(xgb_auc):.4f}")


====== XGBOOST (CV RESULTS) ======
Accuracy : 0.9940 ± 0.0019
ROC-AUC  : 0.9994 ± 0.0006
